# Sarcasm detection with Jev through OpenRouter

This notebook evaluates TypeSafe's Jev classifier through OpenRouter on either the official SARC 2.0 balanced test split or a custom CSV. It downloads only the files required for evaluation, sends a small sample to Jev, checkpoints predictions, and reports accuracy, precision, recall, F1, latency, token usage, and API cost.

The API key is read from `OPENROUTER_API_KEY`, a local `.env` file, or a hidden prompt. It is never written to this notebook or the output CSV, and `.env` is excluded from Git. Review `MAX_EXAMPLES` before running the live cell because OpenRouter pricing can change.

In [ ]:
from pathlib import Path
from getpass import getpass
from urllib import request, error
import bz2
import csv
import hashlib
import json
import os
import random
import time

# ---- Run configuration ----
DATA_SOURCE = "sarc2"       # "sarc2" or "custom_csv"
CUSTOM_CSV_PATH = Path("data/my_data.csv")
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
CONTEXT_COLUMN = "context"  # Optional in a custom CSV

MAX_EXAMPLES = 1000           # Balanced benchmark sample; one API call per example
RANDOM_SEED = 42
JEV_MODEL = "typesafe/jev-1.13"  # Pinned for reproducible results
RUN_LIVE = True               # Set False to prepare/inspect data without API calls
RESUME = True                 # Reuse successful rows already in the checkpoint
REQUEST_DELAY_SECONDS = 0.05
PROGRESS_EVERY = 25           # Keep large runs readable
CHECKPOINT_EVERY = 10         # Save frequently without rewriting after every row

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "sarc2-main"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PREDICTIONS_PATH = OUTPUT_DIR / "jev_predictions.csv"

SARC_BASE_URL = "https://nlp.cs.princeton.edu/old/SARC/2.0/main"
SARC_FILES = {
    "test-balanced.csv.bz2": f"{SARC_BASE_URL}/test-balanced.csv.bz2",
    "comments.json.bz2": f"{SARC_BASE_URL}/comments.json.bz2",
}
OPENROUTER_DECISIONS_URL = "https://openrouter.ai/api/alpha/decisions"

random.seed(RANDOM_SEED)
print(f"Data source: {DATA_SOURCE}; maximum live calls: {MAX_EXAMPLES}; model: {JEV_MODEL}")

## Load the evaluation data

SARC stores split IDs separately from a large compressed comment dictionary. The streaming reader below extracts only the comments needed for this run, so it does not expand the full multi-gigabyte JSON file in memory or on disk.

In [ ]:
def download_file(url, destination, chunk_size=1024 * 1024):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        print(f"Using {destination} ({destination.stat().st_size / 1024**2:.1f} MiB)")
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix + ".part")
    print(f"Downloading {url}")
    with request.urlopen(url, timeout=120) as response, partial.open("wb") as output:
        total = int(response.headers.get("Content-Length", 0))
        downloaded = 0
        while True:
            chunk = response.read(chunk_size)
            if not chunk:
                break
            output.write(chunk)
            downloaded += len(chunk)
            if total and downloaded % (25 * chunk_size) < chunk_size:
                print(f"  {100 * downloaded / total:5.1f}%")
    partial.replace(destination)
    return destination


def iter_compressed_json_object(path, chunk_size=1024 * 1024):
    """Yield (key, value) pairs from a bz2-compressed top-level JSON object."""
    decoder = json.JSONDecoder()
    with bz2.open(path, "rt", encoding="utf-8") as stream:
        buffer, position, eof = "", 0, False

        def read_more():
            nonlocal buffer, eof
            chunk = stream.read(chunk_size)
            if chunk:
                buffer += chunk
            else:
                eof = True

        def skip_space():
            nonlocal position
            while True:
                while position < len(buffer) and buffer[position].isspace():
                    position += 1
                if position < len(buffer) or eof:
                    return
                read_more()

        read_more()
        skip_space()
        if position >= len(buffer) or buffer[position] != "{":
            raise ValueError("Expected a top-level JSON object")
        position += 1

        while True:
            if position > chunk_size:
                buffer = buffer[position:]
                position = 0
            skip_space()
            if position < len(buffer) and buffer[position] == "}":
                return

            while True:
                try:
                    key, end = decoder.raw_decode(buffer, position)
                    break
                except json.JSONDecodeError:
                    if eof:
                        raise
                    read_more()
            position = end
            skip_space()
            if position >= len(buffer) or buffer[position] != ":":
                raise ValueError(f"Expected ':' after JSON key {key!r}")
            position += 1
            skip_space()

            while True:
                try:
                    value, end = decoder.raw_decode(buffer, position)
                    break
                except json.JSONDecodeError:
                    if eof:
                        raise
                    read_more()
            position = end
            yield key, value
            skip_space()
            if position < len(buffer) and buffer[position] == ",":
                position += 1
            elif position < len(buffer) and buffer[position] == "}":
                return
            elif eof:
                raise ValueError("Unexpected end of JSON object")


def normalize_label(value):
    label = str(value).strip().lower()
    positive = {"1", "true", "yes", "sarcastic", "sarcasm"}
    negative = {"0", "false", "no", "not_sarcastic", "not sarcastic", "literal"}
    if label in positive:
        return 1
    if label in negative:
        return 0
    raise ValueError(f"Unsupported label {value!r}; use 0/1 or not_sarcastic/sarcastic")


def load_sarc_records(limit):
    for filename, url in SARC_FILES.items():
        download_file(url, DATA_DIR / filename)

    specs = []
    split_path = DATA_DIR / "test-balanced.csv.bz2"
    with bz2.open(split_path, "rt", encoding="utf-8", newline="") as source:
        for row_number, row in enumerate(csv.reader(source, delimiter="|"), start=1):
            if len(row) != 3:
                raise ValueError(f"Unexpected SARC row {row_number}: {row!r}")
            context_ids = row[0].split()
            reply_ids = row[1].split()
            labels = row[2].split()
            if len(reply_ids) != len(labels):
                raise ValueError(f"Mismatched replies and labels on SARC row {row_number}")
            for reply_id, label in zip(reply_ids, labels):
                specs.append({
                    "row_id": f"sarc-{row_number}-{reply_id}",
                    "context_ids": context_ids,
                    "reply_id": reply_id,
                    "label": normalize_label(label),
                })
                if len(specs) >= limit:
                    break
            if len(specs) >= limit:
                break

    required_ids = {item for spec in specs for item in spec["context_ids"]}
    required_ids.update(spec["reply_id"] for spec in specs)
    comments = {}
    for comment_id, payload in iter_compressed_json_object(DATA_DIR / "comments.json.bz2"):
        if comment_id in required_ids:
            comments[comment_id] = payload.get("text", "")
            if len(comments) == len(required_ids):
                break
    missing = required_ids.difference(comments)
    if missing:
        raise KeyError(f"Could not find {len(missing)} required SARC comments")

    records = []
    for spec in specs:
        context = " \n".join(comments[item] for item in spec["context_ids"] if comments[item])
        records.append({
            "row_id": spec["row_id"],
            "context": context,
            "text": comments[spec["reply_id"]],
            "label": spec["label"],
        })
    return records


def load_custom_records(path, limit):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Custom CSV not found: {path}. It needs '{TEXT_COLUMN}' and '{LABEL_COLUMN}' columns."
        )
    records = []
    with path.open("r", encoding="utf-8-sig", newline="") as source:
        reader_ = csv.DictReader(source)
        required = {TEXT_COLUMN, LABEL_COLUMN}
        if not required.issubset(reader_.fieldnames or []):
            raise ValueError(f"CSV columns must include {sorted(required)}; found {reader_.fieldnames}")
        for row_number, row in enumerate(reader_, start=1):
            records.append({
                "row_id": f"custom-{row_number}",
                "context": row.get(CONTEXT_COLUMN, "") or "",
                "text": row[TEXT_COLUMN],
                "label": normalize_label(row[LABEL_COLUMN]),
            })
            if len(records) >= limit:
                break
    return records


records = (
    load_sarc_records(MAX_EXAMPLES)
    if DATA_SOURCE == "sarc2"
    else load_custom_records(CUSTOM_CSV_PATH, MAX_EXAMPLES)
)

counts = {0: sum(row["label"] == 0 for row in records), 1: sum(row["label"] == 1 for row in records)}
print(f"Prepared {len(records)} examples: {counts[1]} sarcastic, {counts[0]} not sarcastic")
for row in records[:3]:
    preview = row["text"].replace("\n", " ")[:140]
    print(f"[{row['label']}] {preview}")

## Call Jev

Each request asks Jev to choose exactly one of two labels. Context and the reply are sent as separate fields so the model knows which text to classify. HTTP 429, 529, and transient server errors are retried with exponential backoff.

In [ ]:
def read_dotenv_value(path, name):
    path = Path(path)
    if not path.exists():
        return ""
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("export "):
            line = line[7:].lstrip()
        key_name, separator, value = line.partition("=")
        if separator and key_name.strip() == name:
            value = value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
                value = value[1:-1]
            return value.strip()
    return ""


def get_api_key():
    key = os.environ.get("OPENROUTER_API_KEY", "").strip()
    if not key:
        key = read_dotenv_value(PROJECT_ROOT / ".env", "OPENROUTER_API_KEY")
    if not key:
        key = getpass("OpenRouter API key (hidden, not saved): " ).strip()
    if not key:
        raise RuntimeError("An OpenRouter API key is required for a live run")
    return key


def openrouter_decision(api_key, payload, attempts=4, timeout=60):
    body = json.dumps(payload).encode("utf-8")
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Accept": "application/json",
        "Content-Type": "application/json",
        "X-Title": "Sarcasm Detection Jev Evaluation",
    }
    retryable = {429, 500, 502, 503, 504, 529}

    for attempt in range(attempts):
        req = request.Request(OPENROUTER_DECISIONS_URL, data=body, headers=headers, method="POST")
        try:
            with request.urlopen(req, timeout=timeout) as response:
                return json.loads(response.read().decode("utf-8"))
        except error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")[:500]
            if exc.code not in retryable or attempt == attempts - 1:
                raise RuntimeError(f"OpenRouter HTTP {exc.code}: {detail}") from exc
            retry_after = exc.headers.get("Retry-After")
            delay = float(retry_after) if retry_after else 2 ** attempt
        except (error.URLError, TimeoutError) as exc:
            if attempt == attempts - 1:
                raise RuntimeError(f"OpenRouter request failed: {exc}") from exc
            delay = 2 ** attempt
        time.sleep(delay + random.random() * 0.25)


def classify_sarcasm(record, api_key):
    payload = {
        "model": JEV_MODEL,
        "state": {
            "conversation_context": record["context"],
            "reply_to_classify": record["text"],
        },
        "questions": {
            "sarcasm_label": {
                "type": "choice",
                "instructions": (
                    "Classify reply_to_classify. Use conversation_context only as supporting context. "
                    "Choose sarcastic only when the reply is intended sarcastically or ironically."
                ),
                "criteria": {
                    "sarcastic": (
                        "The reply conveys sarcasm or verbal irony: its intended meaning or attitude "
                        "contrasts with the literal wording, often to mock, criticize, or joke."
                    ),
                    "not_sarcastic": (
                        "The reply is literal or sincere; humor, disagreement, exaggeration, or negativity "
                        "alone do not make it sarcastic."
                    ),
                },
            }
        },
    }
    started = time.perf_counter()
    response = openrouter_decision(api_key, payload)
    elapsed = time.perf_counter() - started
    answer = response["answers"]["sarcasm_label"]
    choice = answer["choice"]
    return {
        "predicted_label": 1 if choice == "sarcastic" else 0,
        "predicted_name": choice,
        "sarcastic_probability": answer["probabilities"].get("sarcastic", ""),
        "confidence": answer.get("confidence", ""),
        "latency_seconds": round(elapsed, 4),
        "input_tokens": response.get("usage", {}).get("input_tokens", ""),
        "output_tokens": response.get("usage", {}).get("output_tokens", ""),
        "cost_usd": response.get("usage", {}).get("cost", ""),
        "model": response.get("model", JEV_MODEL),
        "provider": response.get("provider", ""),
    }


RESULT_FIELDS = [
    "sample_id", "row_id", "true_label", "predicted_label", "predicted_name",
    "sarcastic_probability", "confidence", "latency_seconds",
    "input_tokens", "output_tokens", "cost_usd", "model", "provider", "error",
]


def sample_id(record):
    content = f"{record['context']}\0{record['text']}".encode("utf-8")
    return hashlib.sha256(content).hexdigest()[:16]


def read_checkpoint(path):
    if not Path(path).exists():
        return []
    with Path(path).open("r", encoding="utf-8", newline="") as source:
        return list(csv.DictReader(source))


def write_checkpoint(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as output:
        writer = csv.DictWriter(output, fieldnames=RESULT_FIELDS)
        writer.writeheader()
        writer.writerows(rows)

In [ ]:
if not RUN_LIVE:
    print("RUN_LIVE is False. Data is ready, but no API calls were made.")
    results = read_checkpoint(PREDICTIONS_PATH)
else:
    api_key = get_api_key()
    print(f"Using OpenRouter Decisions API with {JEV_MODEL}.")

    old_results = read_checkpoint(PREDICTIONS_PATH) if RESUME else []
    reusable = {row["sample_id"]: row for row in old_results if not row.get("error")}
    results = []

    for index, record in enumerate(records, start=1):
        sid = sample_id(record)
        if sid in reusable:
            results.append(reusable[sid])
            if index <= 5 or index % PROGRESS_EVERY == 0 or index == len(records):
                print(f"[{index:>4}/{len(records)}] reused {record['row_id']}")
            continue

        base = {
            "sample_id": sid,
            "row_id": record["row_id"],
            "true_label": record["label"],
        }
        try:
            result = {**base, **classify_sarcasm(record, api_key), "error": ""}
            if index <= 5 or index % PROGRESS_EVERY == 0 or index == len(records):
                print(
                    f"[{index:>4}/{len(records)}] true={record['label']} "
                    f"pred={result['predicted_label']} p={float(result['sarcastic_probability']):.3f} "
                    f"({result['latency_seconds']:.3f}s)"
                )
        except Exception as exc:
            result = {**base, **{field: "" for field in RESULT_FIELDS if field not in base}, "error": str(exc)}
            print(f"[{index:>4}/{len(records)}] ERROR: {exc}")
        results.append(result)
        if index % CHECKPOINT_EVERY == 0:
            write_checkpoint(PREDICTIONS_PATH, results)
        time.sleep(REQUEST_DELAY_SECONDS)

    write_checkpoint(PREDICTIONS_PATH, results)
    print(f"Saved {len(results)} rows to {PREDICTIONS_PATH}")

## Evaluation

Metrics are calculated only for successful API responses. In this task, label `1` means sarcastic and label `0` means not sarcastic.

In [ ]:
successful = [row for row in results if row.get("predicted_label") not in (None, "") and not row.get("error")]
if not successful:
    print("No successful predictions to evaluate yet. Run the live classification cell first.")
else:
    y_true = [int(row["true_label"]) for row in successful]
    y_pred = [int(row["predicted_label"]) for row in successful]
    tp = sum(actual == 1 and predicted == 1 for actual, predicted in zip(y_true, y_pred))
    tn = sum(actual == 0 and predicted == 0 for actual, predicted in zip(y_true, y_pred))
    fp = sum(actual == 0 and predicted == 1 for actual, predicted in zip(y_true, y_pred))
    fn = sum(actual == 1 and predicted == 0 for actual, predicted in zip(y_true, y_pred))
    accuracy = (tp + tn) / len(successful)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    latencies = [float(row["latency_seconds"]) for row in successful if row.get("latency_seconds") not in (None, "")]
    input_tokens = sum(int(row["input_tokens"]) for row in successful if row.get("input_tokens") not in (None, ""))
    output_tokens = sum(int(row["output_tokens"]) for row in successful if row.get("output_tokens") not in (None, ""))
    total_cost = sum(float(row["cost_usd"]) for row in successful if row.get("cost_usd") not in (None, ""))

    print(f"Evaluated: {len(successful)} / {len(results)}")
    print(f"Accuracy : {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall   : {recall:.3f}")
    print(f"F1       : {f1:.3f}")
    print("\nConfusion matrix (rows=true, columns=predicted)")
    print("                 pred 0   pred 1")
    print(f"true 0           {tn:>6}   {fp:>6}")
    print(f"true 1           {fn:>6}   {tp:>6}")
    if latencies:
        print(f"\nMean latency: {sum(latencies) / len(latencies):.3f}s")
    print(f"Token usage: {input_tokens} input, {output_tokens} output")
    print(f"OpenRouter cost reported by API: ${total_cost:.8f}")

## Latest completed run

On 2026-10-02, the notebook classified 1,000 balanced SARC examples through OpenRouter using `typesafe/jev-1.13` (served snapshot `typesafe/jev-1.13-20260917`). All 1,000 calls succeeded.

- Accuracy: **0.589**
- Precision: **0.604**
- Recall: **0.516**
- F1: **0.557**
- Confusion matrix: **TN 331, FP 169, FN 242, TP 258**
- Mean latency: **0.155 seconds**
- Token usage: **445,871 input / 43,146 output**
- OpenRouter cost reported by the API: **$0.01872658**

The row-level results are in `outputs/jev_predictions.csv`. Re-running with `RESUME = True` reuses these successful predictions instead of paying for them again.

## Using your own data

Create `data/my_data.csv` with at least `text` and `label` columns; an optional `context` column supplies conversation context. Accepted labels are `0`/`1`, `not_sarcastic`/`sarcastic`, `false`/`true`, or `no`/`yes`. Then set `DATA_SOURCE = "custom_csv"` in the configuration cell and run all cells again.

The predictions checkpoint intentionally excludes raw text, reducing the chance that private input data is duplicated into output artifacts.